In [1]:
import pandas as pd
import numpy as np
import os

RAW_DATA_PATH = "../data/raw/"

files = {
    "demographics"  : "DEMO_J.XPT",
    "body_measures" : "BMX_J.XPT",
    "hba1c"         : "GHB_J.XPT",
    "diabetes_q"    : "DIQ_J.XPT",
    "blood_pressure": "BPX_J.XPT",
    "cholesterol"   : "TCHOL_J.XPT"
}

dataframes = {}
for name, filename in files.items():
    filepath = os.path.join(RAW_DATA_PATH, filename)
    df = pd.read_sas(filepath, format='xport', encoding='utf-8')
    dataframes[name] = df

print("✅ All datasets loaded!")

c:\Users\vsmuh\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


✅ All datasets loaded!


In [2]:
# Select only the columns we need and rename them

demo = dataframes['demographics'][['SEQN','RIDAGEYR','RIAGENDR','RIDRETH3','INDFMPIR']].copy()
demo.columns = ['participant_id','age','gender','ethnicity','poverty_ratio']

bmi = dataframes['body_measures'][['SEQN','BMXBMI','BMXWAIST']].copy()
bmi.columns = ['participant_id','bmi','waist_cm']

hba1c = dataframes['hba1c'][['SEQN','LBXGH']].copy()
hba1c.columns = ['participant_id','hba1c']

diab_q = dataframes['diabetes_q'][['SEQN','DIQ010']].copy()
diab_q.columns = ['participant_id','diabetes_self_report']

bp = dataframes['blood_pressure'][['SEQN','BPXSY1','BPXDI1']].copy()
bp.columns = ['participant_id','systolic_bp','diastolic_bp']

chol = dataframes['cholesterol'][['SEQN','LBXTC']].copy()
chol.columns = ['participant_id','total_cholesterol']

print("✅ Columns selected and renamed!")
print(f"Demo columns    : {list(demo.columns)}")
print(f"BMI columns     : {list(bmi.columns)}")
print(f"HbA1c columns   : {list(hba1c.columns)}")
print(f"Diabetes columns: {list(diab_q.columns)}")
print(f"BP columns      : {list(bp.columns)}")
print(f"Cholesterol cols: {list(chol.columns)}")

✅ Columns selected and renamed!
Demo columns    : ['participant_id', 'age', 'gender', 'ethnicity', 'poverty_ratio']
BMI columns     : ['participant_id', 'bmi', 'waist_cm']
HbA1c columns   : ['participant_id', 'hba1c']
Diabetes columns: ['participant_id', 'diabetes_self_report']
BP columns      : ['participant_id', 'systolic_bp', 'diastolic_bp']
Cholesterol cols: ['participant_id', 'total_cholesterol']


In [3]:
# Merge all tables on participant_id (SEQN)
df = demo.copy()

for table in [bmi, hba1c, diab_q, bp, chol]:
    df = df.merge(table, on='participant_id', how='inner')

print(f"✅ Merged dataset shape: {df.shape}")
print(f"\nFirst look:")
print(df.head())

✅ Merged dataset shape: (6401, 12)

First look:
   participant_id   age  gender  ethnicity  poverty_ratio   bmi  waist_cm  \
0         93705.0  66.0     2.0        4.0           0.82  31.7     101.8   
1         93706.0  18.0     1.0        6.0            NaN  21.5      79.3   
2         93707.0  13.0     1.0        7.0           1.88  18.1      64.1   
3         93708.0  66.0     2.0        6.0           1.63  23.7      88.2   
4         93709.0  75.0     2.0        4.0           0.41  38.9     113.0   

   hba1c  diabetes_self_report  systolic_bp  diastolic_bp  total_cholesterol  
0    6.2                   2.0          NaN           NaN              157.0  
1    5.2                   2.0        112.0          74.0              148.0  
2    5.6                   2.0        128.0          38.0              189.0  
3    6.2                   3.0          NaN           NaN              209.0  
4    6.3                   2.0        120.0          66.0              176.0  


In [4]:
# 1. Remove refused/unknown codes
df = df[df['diabetes_self_report'] != 9.0]

# 2. Replace gender codes with labels
df['gender'] = df['gender'].map({1.0: 'Male', 2.0: 'Female'})

# 3. Replace ethnicity codes with labels
ethnicity_map = {
    1.0: 'Mexican American',
    2.0: 'Other Hispanic',
    3.0: 'Non-Hispanic White',
    4.0: 'Non-Hispanic Black',
    6.0: 'Non-Hispanic Asian',
    7.0: 'Other'
}
df['ethnicity'] = df['ethnicity'].map(ethnicity_map)

# 4. Replace diabetes self report codes
diabetes_map = {1.0: 'Yes', 2.0: 'No', 3.0: 'Borderline'}
df['diabetes_self_report'] = df['diabetes_self_report'].map(diabetes_map)

# 5. Check missing values
print("Missing values per column:")
print(df.isnull().sum())
print(f"\nDataset shape before dropping nulls: {df.shape}")

# 6. Drop remaining missing values
df = df.dropna()
print(f"Dataset shape after dropping nulls : {df.shape}")

Missing values per column:
participant_id            0
age                       0
gender                    0
ethnicity                 0
poverty_ratio           821
bmi                     107
waist_cm                376
hba1c                   356
diabetes_self_report      0
systolic_bp             726
diastolic_bp            726
total_cholesterol       468
dtype: int64

Dataset shape before dropping nulls: (6397, 12)
Dataset shape after dropping nulls : (4478, 12)


In [5]:
# Create diabetes status based on HbA1c clinical thresholds
def classify_diabetes(hba1c):
    if hba1c < 5.7:
        return 0   # Normal
    elif hba1c < 6.5:
        return 1   # Pre-diabetic
    else:
        return 2   # Diabetic

df['diabetes_status'] = df['hba1c'].apply(classify_diabetes)

# Label map for reference
label_map = {0: 'Normal', 1: 'Pre-diabetic', 2: 'Diabetic'}

print("✅ Target variable created!")
print("\nClass distribution:")
print(df['diabetes_status'].value_counts().map(lambda x: x).to_frame())
print("\nFinal clean dataset shape:", df.shape)
print("\nColumn names:", list(df.columns))

# Save cleaned data
df.to_csv('../data/processed/nhanes_cleaned.csv', index=False)
print("\n✅ Cleaned dataset saved to data/processed/nhanes_cleaned.csv")

✅ Target variable created!

Class distribution:
                 count
diabetes_status       
0                 2775
1                 1177
2                  526

Final clean dataset shape: (4478, 13)

Column names: ['participant_id', 'age', 'gender', 'ethnicity', 'poverty_ratio', 'bmi', 'waist_cm', 'hba1c', 'diabetes_self_report', 'systolic_bp', 'diastolic_bp', 'total_cholesterol', 'diabetes_status']

✅ Cleaned dataset saved to data/processed/nhanes_cleaned.csv
